# 5 — Banded Recommender (multimedia posts + marketplace products)

Same band protocol applied to recommendation: N two-tower / sequential recommenders train on
different slices & seeds (collaborative, content-multimedia, sequential). Members join
HR@10 bands `[37..81..90]%`, wait the grace window, then bag (rank-fusion: mean reciprocal-rank
of member rankings). Isolating before fusion is what keeps niche tastes (long-tail products,
new creators) from being drowned by the popular head.

**BuddyUp mapping:** `feed_ranking.ipynb` (posts) + `matching_embeddings.ipynb` (two-tower +
FAISS) + `apps/marketplace` products. This notebook is the ensemble wrapper around those two.

In [1]:
import importlib.util, os, pathlib, sys
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None: break
    p = p.parent
if ai is None: raise RuntimeError('ai_service not found')
sys.path.insert(0, str(ai / 'training')); sys.path.insert(0, str(ai))
os.chdir(ai / 'notebooks')
# ── load the repo .env (BUDDY_SCALE, KAGGLE_API_TOKEN, …) BEFORE bootstrap ──
# bootstrap reads BUDDY_SCALE at import time, so this must run first. Uses
# python-dotenv when available, else a tiny built-in parser (Kaggle-safe).
def _find_dotenv(start):
    p = pathlib.Path(start).resolve()
    while p != p.parent:
        f = p / '.env'
        if f.is_file():
            return f
        p = p.parent
    return None

_env_file = _find_dotenv(ai)
try:
    from dotenv import load_dotenv
    load_dotenv(_env_file)
except ImportError:
    if _env_file:
        for _line in _env_file.read_text().splitlines():
            _line = _line.strip()
            if not _line or _line.startswith('#') or '=' not in _line:
                continue
            _k, _, _v = _line.partition('=')
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))
print('[env]', _env_file or 'no .env found', '| BUDDY_SCALE =', os.environ.get('BUDDY_SCALE'),
      '| KAGGLE_API_TOKEN =', 'set' if os.environ.get('KAGGLE_API_TOKEN') else 'missing')
_missing = [m for m in ['torch'] if importlib.util.find_spec(m) is None]
if _missing:
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(_missing))
# bootstrap.py reads BUDDY_SCALE at import time; if an earlier run in this
# same kernel cached the module (e.g. before the .env was loaded), drop the
# stale copy so the current environment is honoured.
for _stale in ('training.bootstrap', 'bootstrap'):
    _m = sys.modules.get(_stale)
    if _m is not None and getattr(_m, 'BUDDY_SCALE', None) != os.environ.get('BUDDY_SCALE'):
        sys.modules.pop(_stale, None)
        print(f'[bootstrap] re-importing {_stale} (stale scale cache cleared)')
try:
    from training.bootstrap import *
    CFG = init(scale=os.environ.get('BUDDY_SCALE') or None)
except Exception as e:
    print('[bootstrap] unavailable:', e); CFG = {}
except Exception as e:
    print('[bootstrap] unavailable:', e); CFG = {}
SCALE = CFG.get('scale', os.environ.get('BUDDY_SCALE', 'demo'))
print('scale:', SCALE)

[env] /home/peter/Desktop/Buddy-Up/backend/.env | BUDDY_SCALE = demo | KAGGLE_API_TOKEN = set


2026-09-16 16:11:56.114772: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-16 16:11:56.266847: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-09-16 16:11:59.849387: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


scale: demo


In [2]:
import torch, torch.nn as nn, numpy as np
torch.manual_seed(3); np.random.seed(3)
BANDS = [0.37, 0.47, 0.57, 0.67, 0.71, 0.73, 0.77, 0.79, 0.81, 0.85, 0.90]
GRACE = {'smoke': 1, 'demo': 2, 'full': 4}[SCALE]
N = {'smoke': 4, 'demo': 6, 'full': 12}[SCALE]
NU, NI, D = 300, 800, 32  # users, items (posts+products), dim
DL = 8  # true latent dim of the synthetic world (smaller = easier factor recovery)

class TwoTower(nn.Module):
    def __init__(self, nu, ni, d, seed=0, content_dim=0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.u = nn.Embedding(nu, d); self.i = nn.Embedding(ni, d)
        nn.init.normal_(self.u.weight, std=d ** -0.5, generator=g); nn.init.normal_(self.i.weight, std=d ** -0.5, generator=g)
        self.use_content = content_dim > 0
        if self.use_content: self.cf = nn.Linear(content_dim, d)  # multimedia content vector
    def score(self, u, items, cfeat=None):
        s = self.u(u) @ self.i(items).T
        if self.use_content and cfeat is not None: s = s + (self.u(u) @ self.cf(cfeat).T)
        return s

# Synthetic interactions. Real: Amazon Reviews / MovieLens / BuddyUp engagement.csv.
rng = np.random.default_rng(7)
UL = rng.normal(size=(NU, DL)); IL = rng.normal(size=(NI, DL))
ULt = torch.tensor(UL, dtype=torch.float32); ILt = torch.tensor(IL, dtype=torch.float32)  # oracle for eval
pairs = [(int(rng.integers(NU)), int(rng.integers(NI))) for _ in range(20000)]
labels = [float(rng.random() < 1/(1+np.exp(-(UL[u] @ IL[i])))) for u, i in pairs]
R = rng.normal(size=(DL, 16))
C = torch.tensor(IL @ R + 0.5 * rng.normal(size=(NI, 16)), dtype=torch.float32)
# content carries true-factor signal (like real CLIP/taste embeddings): hybrids generalize
print('pairs:', len(pairs), 'pos-rate:', round(float(np.mean(labels)), 3))
# --- real batch (BUDDY_BATCH) overrides synthetic when present ---
from batch_data import has_batch, batch_meta, load_tensors
_VALPOS = None
if has_batch():
    _m = batch_meta(); _b = load_tensors('U', 'I', 'y', 'val_u', 'val_i', 'C')
    NU, NI = _m['n_users'], _m['n_items']; C = _b['C']
    pairs = list(zip(_b['U'].tolist(), _b['I'].tolist())); labels = _b['y'].tolist()
    _VALPOS = {}
    for _u, _i in zip(_b['val_u'].tolist(), _b['val_i'].tolist()):
        _VALPOS.setdefault(_u, []).append(_i)
    print(f"REAL batch: {len(pairs)} pairs, {NU} users x {NI} items", "|", _m["source"])

pairs: 20000 pos-rate: 0.496
REAL batch: 273923 pairs, 1500 users x 8000 items | food.com ratings>=4 pos, recipe-name content


In [3]:
# Train members on bootstrapped slices with mixed configs (collab-only vs content-hybrid)
import random
members = [TwoTower(NU, NI, D, seed=i, content_dim=(16 if i % 2 else 0)) for i in range(N)]
opts = [torch.optim.Adam(m.parameters(), lr=5e-3) for m in members]
bce = nn.BCEWithLogitsLoss()
EPOCHS = {'smoke': 2, 'demo': 5, 'full': 12}[SCALE]
def hr_at_k(m, k=10, trials=200):
    m.eval(); hits = 0
    with torch.no_grad():
        for _ in range(trials):
            u = int(rng.integers(NU))
            if _VALPOS is not None:  # REAL: held-out liked item
                while u not in _VALPOS: u = int(rng.integers(NU))
                pos = int(rng.choice(_VALPOS[u]))
            else:
                pool = torch.tensor(rng.integers(0, NI, 200))  # oracle picks a truly-liked pos
                pos = int(pool[(ULt[u] @ ILt[pool].T).argmax()])
            cands = [pos] + [int(rng.integers(NI)) for _ in range(49)]
            ci = torch.tensor(cands)
            s = m.score(torch.tensor([u]), ci, C[ci] if m.use_content else None)[0]
            hits += int(torch.topk(s, k).indices.eq(0).any())
    return hits / trials
band, joined, best = {}, {}, {}
UPDATES = min(150, max(8, int(12 * len(pairs) / 4096 / max(1, EPOCHS))))  # ~12 passes over pairs
for ep in range(EPOCHS):
    for mi, m in enumerate(members):
        m.train()
        for _ in range(UPDATES):
            idx = np.random.choice(len(pairs), 4096)
            uu = torch.tensor([pairs[j][0] for j in idx]); ii = torch.tensor([pairs[j][1] for j in idx])
            yy = torch.tensor([labels[j] for j in idx]).float()
            opts[mi].zero_grad()
            s = (m.u(uu) * m.i(ii)).sum(-1)
            if m.use_content: s = s + (m.u(uu) * m.cf(C[ii])).sum(-1)
            loss = bce(s, yy); loss.backward(); opts[mi].step()
    print(f'--- epoch {ep} ---')
    for mi, m in enumerate(members):
        hr = hr_at_k(m); best[mi] = max(hr, best.get(mi, 0.))
        b = max([t for t in BANDS if best[mi] >= t], default=None)
        if b and (mi not in joined or b > joined[mi][0]):
            joined[mi] = (b, ep); print(f'  member {mi} HR@10={best[mi]:.3f} joins {int(b*100)}%')
print('joined:', {k: (int(v[0]*100), v[1]) for k, v in joined.items()})

--- epoch 0 ---


--- epoch 1 ---


--- epoch 2 ---


--- epoch 3 ---


  member 4 HR@10=0.380 joins 37%


--- epoch 4 ---


  member 1 HR@10=0.385 joins 37%


  member 3 HR@10=0.395 joins 37%


joined: {4: (37, 3), 1: (37, 4), 3: (37, 4)}


In [4]:
# Rank-fusion bag over grace-eligible members (mean reciprocal rank)
elig = [mi for mi, (_, e0) in joined.items() if EPOCHS - e0 >= GRACE] or list(range(N))
print('eligible for fusion:', elig)
def fused_hr(trials=200, k=10):
    hits = 0
    with torch.no_grad():
        for _ in range(trials):
            u = int(rng.integers(NU))
            if _VALPOS is not None:  # REAL: held-out liked item
                while u not in _VALPOS: u = int(rng.integers(NU))
                pos = int(rng.choice(_VALPOS[u]))
            else:
                pool = torch.tensor(rng.integers(0, NI, 200))
                pos = int(pool[(ULt[u] @ ILt[pool].T).argmax()])
            cands = [pos] + [int(rng.integers(NI)) for _ in range(49)]
            ci = torch.tensor(cands); mrr = torch.zeros(50)
            for mi in elig:
                m = members[mi]
                s = m.score(torch.tensor([u]), ci, C[ci] if m.use_content else None)[0]
                rank = s.argsort(descending=True).argsort().float() + 1
                mrr += 1.0 / rank
            hits += int(torch.topk(mrr, k).indices.eq(0).any())
    return hits / trials
print('fused HR@10:', round(fused_hr(), 3))
# Export best single tower (item tower -> FAISS pattern in matching_embeddings.ipynb)
bi = int(max(best, key=best.get)); members[bi].eval()
torch.onnx.export(members[bi].i, torch.tensor([0]), '../models/recsys_item_emb.onnx',
    input_names=['item_id'], output_names=['emb'], dynamic_axes={'item_id': {0: 'batch'}})
print('exported ../models/recsys_item_emb.onnx')

eligible for fusion: [4]


fused HR@10: 0.33


/tmp/ipykernel_2869740/580830613.py:27: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(members[bi].i, torch.tensor([0]), '../models/recsys_item_emb.onnx',


[torch.onnx] Obtain model graph for `Embedding(8000, 32)` with `torch.export.export(..., strict=False)`...


[torch.onnx] Obtain model graph for `Embedding(8000, 32)` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
exported ../models/recsys_item_emb.onnx


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


## Data, scrapers vs platforms (notebook 5)

| Need | Recommendation |
|---|---|
| Interaction data (gold) | BuddyUp `engagement.csv` (posts) + marketplace orders/carts. Nothing beats own logs; needs ~10k+ events before beating heuristics. |
| Public bootstrap | Amazon Reviews 2023, MovieLens-25M, Yelp, H&M Fashion Recs (products); MIND (news/posts). Start here for cold-start. |
| Content features | CLIP (image+text), AST/CLAP (audio), VideoMAE (clips) — precompute once, reuse as `C` matrix. |
| Scrapers / bots | Useful ONLY for product-catalog enrichment (price/specs via official APIs: Amazon PA-API, OpenFoodFacts). For posts: never scrape other networks — ToS + copyright. Run a first-party internet agent only for *opt-in* cross-post import (user pastes own link). |
| Platforms (buy-vs-build) | Shaped (rec-API), Tigris/Algolia (search+recs), AWS Personalize / GCP Recommendations AI / Rebuy (hosted recs), Qdrant/Pinecone + FAISS (vector serving). This notebook = the self-hosted alternative; hosted wins if team < 3 ML engineers. |